Simple Raw

In [115]:
import csv

with open('ProtoTaiEtymaRaw.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('ProtoTaiEtymaRaw.csv', 'w', newline='', encoding='utf-8') as out:
    writer = csv.writer(out)

    header = ['No.', 'Gloss', 'PT', 'Siamese', 'Sapa', 'Bao Yen', 'Cao Bang', 'Lungchow', 'Shangsi', 'Yay', 'Saek']
    writer.writerow(header)

    for line in lines[1:]:
        tokens = line.split()
        if not tokens:
            continue

        if tokens[0].endswith('.'):
            section_text = line.strip()
            row = [section_text] + [''] * (len(header) - 1)
            writer.writerow(row)
            continue

        new_line = [tokens[0]]

        # Build gloss
        gloss = tokens[1] if len(tokens) > 1 else ''
        i = 2
        while i < len(tokens):
            if tokens[i].startswith('*'):
                break
            gloss += ' ' + tokens[i]
            i += 1

        new_line.append(gloss)

        # Remaining tokens
        while i < len(tokens):
            if tokens[i].startswith('-'):
                new_line[-1] += ' ' + tokens[i]
                i += 1
                continue
            new_line.append(tokens[i])
            i += 1

        writer.writerow(new_line)

PDFPlumber

In [116]:
from pypdf import PdfReader, PdfWriter
import pdfplumber
import pandas as pd

def rotate_and_extract(input_pdf, start_page, end_page):
    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    for page in reader.pages:
        page.rotate(90)
        writer.add_page(page)

    rotated_path = 'rotated_temp.pdf'
    with open(rotated_path, 'wb') as f:
        writer.write(f)

    dfs = []

    with pdfplumber.open(rotated_path) as pdf:
        for i in range(start_page, end_page + 1):
            page = pdf.pages[i]
            
            table_settings = {
                'vertical_strategy': 'lines',
                'horizontal_strategy': 'lines',
                'snap_tolerance': 4,
                'join_tolerance': 4,
            }
            
            table = page.extract_table(table_settings=table_settings)
            
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])
                df = df.replace('\n', ' ', regex=True)
                df = df.iloc[:, ::3].copy()
                dfs.append(df)

    # combine all pages
    final_df = pd.concat(dfs, ignore_index=True)

    # set headers
    header = ['No.', 'Gloss', 'PT', 'Siamese', 'Sapa', 'Bao Yen', 'Cao Bang', 'Lungchow', 'Shangsi', 'Yay', 'Saek']
    final_df.columns = header

    return final_df


raw_df = rotate_and_extract('ProtoTaiEtyma.pdf', 0, 40)
df = raw_df.copy()
df

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
0,,A. Body parts,,,,,,,,,
1,1,head (1),*krawC,,,,,,lawC1,cawC1,tʰrawC1
2,2,head (2),*truəA,huəA1,hu1,huəA1,tʰuəA1,hu:A1,,,
3,3,head hair,*prɤmA,pʰomA1,pʰumA1,pʰjomA1,pʰjomA1,pʰjumA1,pʰomA1,piəmA1,pʰramA1
4,4,hair knot,*klawC,kla:wC1,,cawC1,cawC1,kjawC1,,,
...,...,...,...,...,...,...,...,...,...,...,...
801,785,which,*ɗaɰA,dajA1,daɰA1,dɤɰA1,dɤɰA1,naɰA2 -i,,,dɤ:A1
802,786,also,*ko:C,kɔ:C1,koC1,,,,,koC2 -t,
803,787,"with, and",*kapD,kapDS1,,kapDS1,kapDS1,,,,kapDS1
804,788,matter,*ɣwa:mA,kʰwa:mA2,,,wa:mA2,va:mA2,,,


In [117]:
rows_with_whitespace = df.iloc[:, df.columns.get_loc('PT'):].apply(
    lambda x: x.str.strip().str.contains(r'\s(?!-)', na=False)
).any(axis=1)
df[rows_with_whitespace]

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
24,24,shoulder,*C̥ .ba:B,ba:B1,ba:B1,ba:B1,ba:B1,ba:B1,ba:B1,baB1,va:B1
26,26,elbow,*C̬ .swo:k D,sɔ:kDL1,soʔDL1,tʰɔ:kDL1,ɬɔkDL1,,,suəkDL2,suəkDL2
29,29,"fingernail, toenail",*C̬ .lepD,lepDS2,lipDS2,lopDS2,lepDS2,lipDS2,lipDS2,ritDS2,li:pDL2
39,39,waist (2),*C̥ .wɯǝtD,,,,,,hutDL1,hɯǝtDL1,vuǝtDS1
50,50,excrement,*C̬ .qɯjC,kʰi:C1,,kʰi:C1,kʰiC1,kʰi:C1,kʰoyC1,hajC2,ɣajC2
...,...,...,...,...,...,...,...,...,...,...,...
781,765,day after tomorrow,*C̬ .rɯ:A,rɯ:nA2 -f,hɯA2,rɯ:A2,lɯA2,lɯ:A2,loyA2,rɯA2,rɯ:A1
786,770,inside,*C̥ .daɰA,najA2 -i,,,dɤɰA1,daɰA1,doyA1,daɰA1,rɤ:A1
789,773,side,*C̥ .bɯǝŋC,bɯǝŋC1,,bɯəŋC1,bɯəŋC1,bɤ:ŋC1,,,viǝŋC1
796,780,not (strong 1),*ɓawB,,,bɤwB1,bɤwB1,bawB1 bo:B1,,bawB1 boB1,bo:B1


In [118]:
df['PT'] = df['PT'].str.replace(r'C̥ ', 'C̥', regex=True)
df['PT'] = df['PT'].str.replace(r'C̬ ', 'C̬', regex=True) 
df['PT'] = df['PT'].str.replace(r'm̩ ', 'm̩', regex=True)

rows_with_whitespace = df.iloc[:, df.columns.get_loc('PT'):].apply(
    lambda x: x.str.strip().str.contains(r'\s(?!-)', na=False)
).any(axis=1)
df[rows_with_whitespace]

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
26,26,elbow,*C̬.swo:k D,sɔ:kDL1,soʔDL1,tʰɔ:kDL1,ɬɔkDL1,,,suəkDL2,suəkDL2
158,154,"tree, wood",*mwajC,ma:jC2,majC2,majC2 -v,maj C2 -v,majC2,majC2,majC2 -i,majC2
164,160,"peel, bark",*plɯəkD,plɯəkDL1,pɯʔDL1,,pɯəkDL1,pɤ:kDL1,,,pla:kDL1 - v
281,273,great-grandchild,*ʰlenC,le:nA1 t,linA1 -t,lɤnA1 -t,lɤnC1,,lɤnC1,,
285,277,mother's younger sibling,*na:C,na:C2,na:C2,na:C2,na:B1 -t,na:B2 t,na:B2 -t,naC2,na:C2
351,341,bamboo tube,*baŋ B/C,,,baŋC1,baŋC1,,,baŋB1,baŋC2
474,462,withered,*ʰriəwB,hiəwB1,hewB1,hɛ:wB1,,he:wB1,hoy B1 -v,rewB1,hɛ:wB1 -i
475,463,dried up,*ʰre:ŋC,hɛ:ŋC1,heŋC1,,,,,re:ŋC1,hɛ:ŋC1 -i - v
796,780,not (strong 1),*ɓawB,,,bɤwB1,bɤwB1,bawB1 bo:B1,,bawB1 boB1,bo:B1
800,784,do not,*ʰɲa:B,ja:B1,,,,ja:B1,,"jiəB1, ja:B1",ja:B1


In [119]:
df[df.isnull().all(axis=1)]

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek


Fix List
1. 26 elbow *C̬.swo:k D: new line
2. 160 peel, bark pla:kDL1 - v: new line
3. 463 dried up hɛ:ŋC1 -i - v: new line
4. Empty rows before each categories

In [120]:
df.to_csv('ProtoTaiEtymaPreprocessed.csv', index=False)

Manual Fixing Inspection

In [20]:
df = pd.read_csv('ProtoTaiEtymaManualFix.csv')
df

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
0,NaN,A. Body parts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,head (1),*krawC,NaN,NaN,NaN,NaN,NaN,lawC1,cawC1,tʰrawC1
2,2.0,head (2),*truəA,huəA1,hu1,huəA1,tʰuəA1,hu:A1,NaN,NaN,NaN
3,3.0,head hair,*prɤmA,pʰomA1,pʰumA1,pʰjomA1,pʰjomA1,pʰjumA1,pʰomA1,piəmA1,pʰramA1
4,4.0,hair knot,*klawC,kla:wC1,NaN,cawC1,cawC1,kjawC1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
792,784.0,do not,*ʰɲa:B,ja:B1,NaN,NaN,NaN,ja:B1,NaN,"jiəB1, ja:B1",ja:B1
793,785.0,which,*ɗaɰA,dajA1,daɰA1,dɤɰA1,dɤɰA1,naɰA2 -i,NaN,NaN,dɤ:A1
794,786.0,also,*ko:C,kɔ:C1,koC1,NaN,NaN,NaN,NaN,koC2 -t,NaN
795,787.0,"with, and",*kapD,kapDS1,NaN,kapDS1,kapDS1,NaN,NaN,NaN,kapDS1


In [21]:
rows_with_whitespace = df.iloc[:, df.columns.get_loc('PT'):].apply(
    lambda x: x.str.strip().str.contains(r'\s(?!-)', na=False)
).any(axis=1)
df[rows_with_whitespace]

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
156,154.0,"tree, wood",*mwajC,ma:jC2,majC2,majC2 -v,maj C2 -v,majC2,majC2,majC2 -i,majC2
277,273.0,great-grandchild,*ʰlenC,le:nA1 t,linA1 -t,lɤnA1 -t,lɤnC1,NaN,lɤnC1,NaN,NaN
281,277.0,mother's younger sibling,*na:C,na:C2,na:C2,na:C2,na:B1 -t,na:B2 t,na:B2 -t,naC2,na:C2
346,341.0,bamboo tube,*baŋ B/C,NaN,NaN,baŋC1,baŋC1,NaN,NaN,baŋB1,baŋC2
468,462.0,withered,*ʰriəwB,hiəwB1,hewB1,hɛ:wB1,NaN,he:wB1,hoy B1 -v,rewB1,hɛ:wB1 -i
788,780.0,not (strong 1),*ɓawB,NaN,NaN,bɤwB1,bɤwB1,bawB1 bo:B1,NaN,bawB1 boB1,bo:B1
792,784.0,do not,*ʰɲa:B,ja:B1,NaN,NaN,NaN,ja:B1,NaN,"jiəB1, ja:B1",ja:B1


In [123]:
df[df.isnull().all(axis=1)]

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek


Observations
1. The remaining whitespaces also exist in the reference material
2. 780 not (strong 1): Lungchow and Yay have variations which are represented by a new line while
3. 784 do not: Yay represents variations by a comma (', ')

Attach Notes to glossary

In [9]:
import pandas as pd

df = pd.read_csv('ProtoTaiEtymaManualFix.csv')
df

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek
0,NaN,A. Body parts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,head (1),*krawC,NaN,NaN,NaN,NaN,NaN,lawC1,cawC1,tʰrawC1
2,2.0,head (2),*truəA,huəA1,hu1,huəA1,tʰuəA1,hu:A1,NaN,NaN,NaN
3,3.0,head hair,*prɤmA,pʰomA1,pʰumA1,pʰjomA1,pʰjomA1,pʰjumA1,pʰomA1,piəmA1,pʰramA1
4,4.0,hair knot,*klawC,kla:wC1,NaN,cawC1,cawC1,kjawC1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
792,784.0,do not,*ʰɲa:B,ja:B1,NaN,NaN,NaN,ja:B1,NaN,"jiəB1, ja:B1",ja:B1
793,785.0,which,*ɗaɰA,dajA1,daɰA1,dɤɰA1,dɤɰA1,naɰA2 -i,NaN,NaN,dɤ:A1
794,786.0,also,*ko:C,kɔ:C1,koC1,NaN,NaN,NaN,NaN,koC2 -t,NaN
795,787.0,"with, and",*kapD,kapDS1,NaN,kapDS1,kapDS1,NaN,NaN,NaN,kapDS1


In [7]:
import re

with open('ProtoTaiNotesToGlossary.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Match: number + dot + space + content (until next number or end)
pattern = r'(\d+)\.\s*(.*?)(?=\n\d+\.|\Z)'

matches = re.findall(pattern, text, re.DOTALL)

notes = pd.DataFrame(matches, columns=['No.', 'Note'])

notes['No.'] = notes['No.'].astype(int)
notes

,No.,Note
0,1,The reflexes of this etymon in NT dialects poi...
1,2,"cf. 頭 tóu ‘head’ (Sagart 1999) from MC dəu, LH..."
2,5,"Reflexes in some dialects, e.g. Ningming /honA..."
3,6,The Siamese form means 'nose brigde'.
4,11,cf. PAN *matá ‘eye’.
...,...,...
293,775,"Some languages unexpectedly have /ɤ/, e.g. Sapa."
294,780,"Many dialects such as Lao, Yuan, Western Nung ..."
295,781,See 'not strong (1)'.
296,782,The tonal irregularity is not surprising becau...


In [10]:
df = df.merge(notes, on='No.', how='left')
df

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek,Note
0,NaN,A. Body parts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,head (1),*krawC,NaN,NaN,NaN,NaN,NaN,lawC1,cawC1,tʰrawC1,The reflexes of this etymon in NT dialects poi...
2,2.0,head (2),*truəA,huəA1,hu1,huəA1,tʰuəA1,hu:A1,NaN,NaN,NaN,"cf. 頭 tóu ‘head’ (Sagart 1999) from MC dəu, LH..."
3,3.0,head hair,*prɤmA,pʰomA1,pʰumA1,pʰjomA1,pʰjomA1,pʰjumA1,pʰomA1,piəmA1,pʰramA1,NaN
4,4.0,hair knot,*klawC,kla:wC1,NaN,cawC1,cawC1,kjawC1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
792,784.0,do not,*ʰɲa:B,ja:B1,NaN,NaN,NaN,ja:B1,NaN,"jiəB1, ja:B1",ja:B1,NaN
793,785.0,which,*ɗaɰA,dajA1,daɰA1,dɤɰA1,dɤɰA1,naɰA2 -i,NaN,NaN,dɤ:A1,NaN
794,786.0,also,*ko:C,kɔ:C1,koC1,NaN,NaN,NaN,NaN,koC2 -t,NaN,Wuming has /kɤC1/. Some dialects have irregula...
795,787.0,"with, and",*kapD,kapDS1,NaN,kapDS1,kapDS1,NaN,NaN,NaN,kapDS1,NaN


In [16]:
df['No.'] = pd.to_numeric(df['No.'], errors='coerce').astype('Int64')
df

,No.,Gloss,PT,Siamese,Sapa,Bao Yen,Cao Bang,Lungchow,Shangsi,Yay,Saek,Note
0,<NA>,A. Body parts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,head (1),*krawC,NaN,NaN,NaN,NaN,NaN,lawC1,cawC1,tʰrawC1,The reflexes of this etymon in NT dialects poi...
2,2,head (2),*truəA,huəA1,hu1,huəA1,tʰuəA1,hu:A1,NaN,NaN,NaN,"cf. 頭 tóu ‘head’ (Sagart 1999) from MC dəu, LH..."
3,3,head hair,*prɤmA,pʰomA1,pʰumA1,pʰjomA1,pʰjomA1,pʰjumA1,pʰomA1,piəmA1,pʰramA1,NaN
4,4,hair knot,*klawC,kla:wC1,NaN,cawC1,cawC1,kjawC1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
792,784,do not,*ʰɲa:B,ja:B1,NaN,NaN,NaN,ja:B1,NaN,"jiəB1, ja:B1",ja:B1,NaN
793,785,which,*ɗaɰA,dajA1,daɰA1,dɤɰA1,dɤɰA1,naɰA2 -i,NaN,NaN,dɤ:A1,NaN
794,786,also,*ko:C,kɔ:C1,koC1,NaN,NaN,NaN,NaN,koC2 -t,NaN,Wuming has /kɤC1/. Some dialects have irregula...
795,787,"with, and",*kapD,kapDS1,NaN,kapDS1,kapDS1,NaN,NaN,NaN,kapDS1,NaN


In [17]:
df.to_csv('ProtoTaiEtymaExact.csv', index=False)